# 17 理解 softmax 和交叉熵

本节目标：理解类别得分、概率分布、softmax 和交叉熵损失。

## 1. 最小概念

分类模型通常先输出每个类别的得分，得分还不是概率。`softmax` 会把得分转换成概率分布：每个概率都大于 0，并且所有类别概率加起来等于 1。

交叉熵用来衡量模型给真实类别的概率有多高：真实类别概率越高，交叉熵越小。

In [ ]:
import torch
import torch.nn.functional as F

torch.set_printoptions(precision=4)


def softmax(scores):
    shifted = scores - scores.max(dim=-1, keepdim=True).values
    exp_scores = torch.exp(shifted)
    return exp_scores / exp_scores.sum(dim=-1, keepdim=True)

scores = torch.tensor([2.0, 1.0, 0.1])
probs = softmax(scores)

print("类别得分:", scores)
print("softmax 概率:", probs)
print("概率加和:", probs.sum())

In [ ]:
batch_scores = torch.tensor([
    [2.0, 1.0, 0.1],
    [0.2, 3.0, 0.5],
])
labels = torch.tensor([0, 1])

batch_probs = softmax(batch_scores)
manual_ce = -torch.log(batch_probs[torch.arange(len(labels)), labels]).mean()
torch_ce = F.cross_entropy(batch_scores, labels)

print("batch scores shape:", tuple(batch_scores.shape))
print("batch probabilities:")
print(batch_probs)
print("每行概率加和:", batch_probs.sum(dim=1))
print("手写交叉熵:", manual_ce.item())
print("PyTorch 交叉熵:", torch_ce.item())

## 小结

softmax 负责把得分变成概率分布；交叉熵负责惩罚“真实类别概率太低”的情况。分类训练通常直接使用 `nn.CrossEntropyLoss`，它内部已经包含了稳定版 softmax 和交叉熵计算。